In [13]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [174]:
import os
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from torch_geometric.data import Data, Batch, separate, InMemoryDataset
from torch_geometric.data import DataLoader
from torch_geometric.utils import to_dense_batch, segment
from torch_geometric.nn import GCNConv, GATConv, GraphConv
from torch.utils.data import IterableDataset
from torch_geometric.datasets import TUDataset

In [175]:
def load_dataset(path):
    ecs = []
    eos = []

    for subject in tqdm(os.listdir(path)):
        for i in range(0, 10):
            try: 
                ec = torch.load(os.path.join(path, subject, 'EC_geometric_{}.pt'.format(i)))
                ecs.append(ec)
            except:
                pass

            try:
                eo = torch.load(os.path.join(path, subject, 'EO_geometric_{}.pt'.format(i)))
                eos.append(eo)
            except:
                pass

            
        '''
        ec = torch.load(os.path.join(path, subject, 'EC_geometric.pt'))
        eo = torch.load(os.path.join(path, subject, 'EO_geometric.pt'))

        ec_indices = len(ec[0].x)
        eo_indices = len(eo[0].x)

        # Break up collated data into individual samples
        # Note that separate is undocumented and I'm not sure what it precisely does
        # but it seems to work and satisfy our use case 


        for i in range(ec_indices):
            ecs.append(separate.separate(cls=ec[0].__class__, batch=ec[0], idx=i, slice_dict=ec[1], decrement=False))

        if eo[1] is None:
            eos.append(eo[0])
        else:
            for i in range(eo_indices):
                eos.append(separate.separate(cls=eo[0].__class__, batch=eo[0], idx=i, slice_dict=eo[1], decrement=False))
        '''
    return ecs + eos

In [181]:
class MyOwnDataset(InMemoryDataset):
    def __init__(self, root, transform=None, pre_transform=None, pre_filter=None):
        super().__init__(root, transform, pre_transform, pre_filter)
        self.load(self.processed_paths[0])

    @property
    def processed_file_names(self):
        return ['data.pt']

    def process(self):
        data_list = [...]

        if self.pre_filter is not None:
            data_list = [data for data in data_list if self.pre_filter(data)]

        if self.pre_transform is not None:
            data_list = [self.pre_transform(data) for data in data_list]

        self.save(data_list, self.processed_paths[0])

In [182]:
datasetList = load_dataset('/home/azureuser/mycontainer/SmallSubset')
print(len(datasetList))

  0%|                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                           | 0/20 [00:00<?, ?it/s]

/tmp/ipykernel_5191/1719766229.py:8: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ec = torch.load(os.path.join(path, subject, 'EC_geometric_{}.pt'.format(i)))
/tmp/ipykerne

80


In [190]:
dataset = CustomDataset(datasetList)

In [191]:
loader = DataLoader(datasetList, batch_size=4, shuffle=True)

/anaconda/envs/EEG-ML/lib/python3.8/site-packages/torch_geometric/deprecation.py:26: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  warnings.warn(out)


In [195]:
print(datasetList[0])
print(loader.dataset[0], loader.dataset[0].edge_index.min(), loader.dataset[0].edge_index.max())
print(loader.dataset[1], loader.dataset[1].edge_index.min(), loader.dataset[1].edge_index.max())
print(loader.dataset[2], loader.dataset[2].edge_index.min(), loader.dataset[2].edge_index.max())
print(loader.dataset[3], loader.dataset[3].edge_index.min(), loader.dataset[3].edge_index.max())
print(loader.dataset[4], loader.dataset[4].edge_index.min(), loader.dataset[4].edge_index.max())
print(loader.dataset[5], loader.dataset[5].edge_index.min(), loader.dataset[5].edge_index.max())
print(loader.dataset[6], loader.dataset[6].edge_index.min(), loader.dataset[6].edge_index.max())
print(loader.dataset[7].is_coalesced())
print(type(loader.dataset[0]))
print(len(loader.dataset))
out = next(iter(loader))
print(out, out.num_graphs)
print(out.to_data_list())
print(Batch.from_data_list(out.to_data_list()))
print(out.edge_index.min(), out.edge_index.max())
print(batch.x[0].shape, batch.x[1].shape, batch.x[2].shape, batch.x[3].shape)

Data(x=[19, 15360], edge_index=[2, 361], edge_attr=[19, 19], y=[0], pos=[19, 3])
Data(x=[19, 15360], edge_index=[2, 361], edge_attr=[19, 19], y=[0], pos=[19, 3]) tensor(0) tensor(18)
Data(x=[19, 15360], edge_index=[2, 361], edge_attr=[19, 19], y=[0], pos=[19, 3]) tensor(0) tensor(18)
Data(x=[19, 15360], edge_index=[2, 361], edge_attr=[19, 19], y=[0], pos=[19, 3]) tensor(0) tensor(18)
Data(x=[19, 15360], edge_index=[2, 361], edge_attr=[19, 19], y=[0], pos=[19, 3]) tensor(0) tensor(18)
Data(x=[19, 15360], edge_index=[2, 361], edge_attr=[19, 19], y=[0], pos=[19, 3]) tensor(0) tensor(18)
Data(x=[19, 15360], edge_index=[2, 361], edge_attr=[19, 19], y=[0], pos=[19, 3]) tensor(0) tensor(18)
Data(x=[19, 15360], edge_index=[2, 361], edge_attr=[19, 19], y=[0], pos=[19, 3]) tensor(0) tensor(18)
True
<class 'torch_geometric.data.data.Data'>
80
DataBatch(x=[4], edge_index=[2, 1444], edge_attr=[4], y=[4], pos=[76, 3], batch=[76], ptr=[5]) 4
[Data(x=[19, 15360], edge_index=[2, 361], edge_attr=[19, 19

In [188]:
class GNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = GCNConv(in_channels=15360, out_channels=15360)

    def forward(self, data):
        x = data.x
        edge_index = data.edge_index
        x = self.conv1(x, edge_index)
        return x
    
GNN_module = GNN()

In [189]:
for data in loader:
    # convert list to tensor
    output = GNN_module(data)
    print(output.shape)
    break

ValueError: 'GCNConv' received a tuple of node features as input while this layer does not support bipartite message passing. Please try other layers such as 'SAGEConv' or 'GraphConv' instead